# Week 81, Day 1: Scalability

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
latencies = [12, 15, 11, 14, 13, 200, 16, 12, 14, 900]   # milliseconds, ten requests
print(sum(latencies) / len(latencies))
ordered = sorted(latencies)
print(ordered[len(ordered) // 2], ordered[-1])
print(sum(1 for x in latencies if x > 100), "of", len(latencies), "were slow")

In [ ]:
import math

def percentile(values, p):
    ordered = sorted(values)
    return ordered[max(1, math.ceil(p / 100 * len(ordered))) - 1]

latencies = [12, 15, 11, 14, 13, 200, 16, 12, 14, 900] * 10
for p in (50, 90, 95, 99):
    print(f"p{p}: {percentile(latencies, p)} ms")

chance_one_slow = 0.01
for calls in (1, 10, 50):
    print(calls, "calls:", f"{1 - (1 - chance_one_slow) ** calls:.0%} of pages hit a slow call")

In [ ]:
dau = 2_000_000
tickets_per_user_per_day = 0.05          # one ticket per user every 20 days
reads_per_ticket = 30                    # page views, list refreshes, API polls
peak_factor = 4
ticket_bytes = 2_000

writes_per_day = dau * tickets_per_user_per_day
avg_write_qps = writes_per_day / 86_400
avg_read_qps = avg_write_qps * reads_per_ticket
print(f"writes: {writes_per_day:,.0f}/day, {avg_write_qps:.1f}/s average, {avg_write_qps * peak_factor:.1f}/s at peak")
print(f"reads: {avg_read_qps:.0f}/s average, {avg_read_qps * peak_factor:.0f}/s at peak")
print(f"storage: {writes_per_day * ticket_bytes * 365 / 1e9:.0f} GB a year")
print(f"read bandwidth at peak: {avg_read_qps * peak_factor * ticket_bytes * 8 / 1e6:.0f} Mbit/s")

In [ ]:
service_ms = 10
mu = 1000 / service_ms                   # 100 requests per second
for rho in (0.5, 0.8, 0.9, 0.95, 0.99):
    lam = rho * mu
    w = 1 / (mu - lam)
    print(f"{rho:.0%} busy: {w * 1000:.0f} ms in the system, {lam * w:.1f} requests inside on average")

> **Key idea.** Latency is a distribution: report p50, p95 and p99 (nearest rank: position ceil(p/100 × n) of the sorted values), because averages hide the tail and pages with many calls hit it often. Estimate before designing: users and actions per day, divided by 86,400 for the average rate, times a peak factor; storage is daily data times retention; bandwidth in bits is bytes × 8; note the read-to-write ratio. Little's law: requests in the system = arrival rate × time in the system. Waiting time explodes as utilisation nears 100% (W = 1/(μ - λ) for a simple queue), so plan for 50 to 70% at peak. Scale up with bigger machines or out with more, which needs stateless services; balance with round robin, weighted, or least connections, plus health checks. The slowest stage is the bottleneck; Amdahl's law limits parallel speed-up to 1/(1 - p).

### Exercise: Percentiles, not averages

Write `percentile(values, p)` with the nearest-rank method (the value at position `ceil(p / 100 * n)` of the sorted values, counting from 1, and at least position 1); raise `ValueError("no values")` for an empty list. Then `latency_report(values)`: a dict with `count`, `mean` (rounded to 1 decimal place), `p50`, `p95`, `p99` and `max`, and `slo_met`: `True` if the p99 is at most 300.

In [ ]:
import math

def percentile(values, p):
    ___

def latency_report(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_percentile():\n    \"nearest rank\"\n    data = [12, 15, 11, 14, 13, 200, 16, 12, 14, 900]\n    assert percentile(data, 50) == 14 and percentile(data, 90) == 200 and percentile(data, 95) == 900\n    assert percentile(list(range(1, 101)), 99) == 99 and percentile(list(range(1, 101)), 1) == 1 and percentile([7], 50) == 7\n    assert percentile(data, 0) == 11\n    try:\n        percentile([], 50)\n    except ValueError as e:\n        assert str(e) == \"no values\"\n    else:\n        raise AssertionError(\"empty\")\n\ndef test_report():\n    \"a summary that shows the tail\"\n    report = latency_report([12, 15, 11, 14, 13, 200, 16, 12, 14, 900] * 10)\n    assert report == {\"count\": 100, \"mean\": 120.7, \"p50\": 14, \"p95\": 900, \"p99\": 900, \"max\": 900, \"slo_met\": False}\n    assert latency_report([100] * 99 + [300])[\"slo_met\"] is True")

### Exercise: Back of the envelope

Write `estimate(dau, actions_per_user, peak_factor, bytes_per_action, read_ratio, years=1)` returning a dict: `writes_per_day` (`dau * actions_per_user`), `write_qps` and `peak_write_qps` (average over 86,400 seconds, and times the peak factor), `peak_read_qps` (peak writes times `read_ratio`), `storage_gb` (writes per day × bytes × 365 × years, in gigabytes of 10^9 bytes) and `peak_read_mbps` (peak reads × bytes × 8, in megabits of 10^6 bits). Round every float to 1 decimal place. Then `size_class(qps)`: `"one server"` below 100, `"a few servers"` below 10,000, otherwise `"a fleet"`.

In [ ]:
def estimate(dau, actions_per_user, peak_factor, bytes_per_action, read_ratio, years=1):
    ___

def size_class(qps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_estimate():\n    \"the helpdesk at scale\"\n    e = estimate(2_000_000, 0.05, 4, 2_000, 30)\n    assert e == {\"writes_per_day\": 100000.0, \"write_qps\": 1.2, \"peak_write_qps\": 4.6, \"peak_read_qps\": 138.9,\n                 \"storage_gb\": 73.0, \"peak_read_mbps\": 2.2}\n    e = estimate(50_000_000, 4, 3, 500, 100, years=5)\n    assert e[\"peak_write_qps\"] == 6944.4 and e[\"peak_read_qps\"] == 694444.4 and e[\"storage_gb\"] == 182500.0\n    assert size_class(4.6) == \"one server\" and size_class(138.9) == \"a few servers\" and size_class(694444.4) == \"a fleet\"\n    assert size_class(100) == \"a few servers\" and size_class(10_000) == \"a fleet\"")

### Exercise: Little's law

Write `in_flight(rate, latency_ms)`: the average number of requests in progress (Little's law, latency converted to seconds). Then `workers_needed(rate, latency_ms, target_utilisation=0.7)`: the number of workers so that each is busy at most `target_utilisation` of the time, rounded **up** (`math.ceil` of in-flight divided by the target; at least 1). And `wait_ms(service_ms, utilisation)`: the average time in a simple single-server queue, `1 / (μ - λ)` in milliseconds, where `μ = 1000 / service_ms` and `λ = utilisation × μ`; raise `ValueError("the queue grows without limit")` for a utilisation of 1 or more. Round the results of `in_flight` and `wait_ms` to 1 decimal place.

In [ ]:
import math

def in_flight(rate, latency_ms):
    ___

def workers_needed(rate, latency_ms, target_utilisation=0.7):
    ___

def wait_ms(service_ms, utilisation):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_little():\n    \"concurrency from rate and time\"\n    assert in_flight(500, 80) == 40.0 and in_flight(3, 2000) == 6.0\n    assert workers_needed(500, 80) == 58 and workers_needed(500, 80, 1.0) == 40 and workers_needed(0.1, 10) == 1\n    assert wait_ms(10, 0.5) == 20.0 and wait_ms(10, 0.9) == 100.0 and wait_ms(10, 0.99) == 1000.0 and wait_ms(4, 0) == 4.0\n    for bad in (1.0, 1.2):\n        try:\n            wait_ms(10, bad)\n        except ValueError as e:\n            assert str(e) == \"the queue grows without limit\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Load balancers

Write three balancers, each with `pick()` returning a backend name. `RoundRobin(backends)`: in turn, wrapping around. `LeastConnections(backends)`: the backend with the fewest open connections (ties go to the earliest in the list); `pick()` also counts a new connection on it, and `done(name)` ends one. `WeightedRoundRobin(weights)` (a dict of name to whole-number weight) using the **smooth** algorithm nginx uses: on each pick, add each backend's weight to its running "current" value, choose the backend with the highest current (ties to the earliest), then subtract the total of all weights from the chosen one's current. All three have `set_healthy(name, healthy)`: unhealthy backends are skipped (and for weighted round robin, left out of the sum and the choice); if none is healthy, `pick()` raises `RuntimeError("no healthy backends")`.

In [ ]:
class RoundRobin:
    ___

class LeastConnections:
    ___

class WeightedRoundRobin:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_round_robin():\n    \"in turn, skipping the unhealthy\"\n    rr = RoundRobin([\"a\", \"b\", \"c\"])\n    assert [rr.pick() for _ in range(5)] == [\"a\", \"b\", \"c\", \"a\", \"b\"]\n    rr.set_healthy(\"c\", False)\n    assert [rr.pick() for _ in range(4)] == [\"a\", \"b\", \"a\", \"b\"]\n    for name in \"ab\":\n        rr.set_healthy(name, False)\n    try:\n        rr.pick()\n    except RuntimeError as e:\n        assert str(e) == \"no healthy backends\"\n    else:\n        raise AssertionError(\"all down\")\n\ndef test_least_connections():\n    \"adapts to slow requests\"\n    lc = LeastConnections([\"a\", \"b\", \"c\"])\n    assert [lc.pick() for _ in range(3)] == [\"a\", \"b\", \"c\"]\n    lc.done(\"b\")\n    assert lc.pick() == \"b\" and lc.pick() == \"a\"\n    lc.set_healthy(\"a\", False)\n    lc.done(\"c\")\n    assert lc.pick() == \"c\"\n\ndef test_weighted():\n    \"smooth weighted round robin\"\n    w = WeightedRoundRobin({\"big\": 5, \"small1\": 1, \"small2\": 1})\n    assert [w.pick() for _ in range(7)] == [\"big\", \"big\", \"small1\", \"big\", \"small2\", \"big\", \"big\"]\n    w2 = WeightedRoundRobin({\"a\": 2, \"b\": 1})\n    picks = [w2.pick() for _ in range(9)]\n    assert picks.count(\"a\") == 6 and picks[:3] == [\"a\", \"b\", \"a\"]\n    w2.set_healthy(\"a\", False)\n    assert [w2.pick() for _ in range(3)] == [\"b\", \"b\", \"b\"]")

### Exercise: Finding the bottleneck

`stages` is a list of `(name, capacity_per_instance, instances)`: each stage can handle `capacity × instances` requests per second. Write `bottleneck(stages)`: `(name, throughput)` of the stage with the lowest total capacity (ties to the earliest). Then `scale_for(stages, target)`: a new list with each stage's instance count raised (never lowered) to the smallest number that handles `target` requests per second at **at most 70% utilisation** (capacity × instances × 0.7 ≥ target). And `amdahl(p, n)`: the speed-up `1 / ((1 - p) + p / n)`, rounded to 2 decimal places.

In [ ]:
import math

def bottleneck(stages):
    ___

def scale_for(stages, target):
    ___

def amdahl(p, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bottleneck():\n    \"the slowest stage limits everything\"\n    stages = [(\"load balancer\", 50_000, 1), (\"app\", 800, 4), (\"cache\", 20_000, 1), (\"database\", 3_000, 1)]\n    assert bottleneck(stages) == (\"database\", 3000)\n    assert bottleneck([(\"a\", 10, 2), (\"b\", 20, 1)]) == (\"a\", 20)\n    scaled = scale_for(stages, 4000)\n    assert scaled == [(\"load balancer\", 50_000, 1), (\"app\", 800, 8), (\"cache\", 20_000, 1), (\"database\", 3_000, 2)]\n    assert bottleneck(scaled) == (\"database\", 6000)\n    assert scale_for([(\"x\", 100, 5)], 10) == [(\"x\", 100, 5)]\n\ndef test_amdahl():\n    \"parallel speed-up has a ceiling\"\n    assert amdahl(0.95, 1) == 1.0 and amdahl(0.95, 10) == 6.9 and amdahl(0.95, 1000) == 19.63 and amdahl(1.0, 8) == 8.0")

### Exercise: Watching a queue fill

Simulate one server, tick by tick, to see utilisation at work. Write `simulate(arrivals, service_per_tick)`: `arrivals` is a list with the number of requests arriving at each tick; each tick, the new arrivals join the queue, then the server finishes up to `service_per_tick` requests from the front. Return `(max_queue, final_queue, served)`, where `max_queue` is the longest queue seen after the server has worked each tick. Then `bursty(n, rate, every)`: a list of `n` ticks where `rate * every` requests arrive on every `every`-th tick (ticks 0, every, 2×every...) and none otherwise.

In [ ]:
def simulate(arrivals, service_per_tick):
    ___

def bursty(n, rate, every):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_simulate():\n    \"steady versus bursty\"\n    assert simulate([5] * 10, 5) == (0, 0, 50)\n    assert simulate([6] * 10, 5) == (10, 10, 50)\n    assert bursty(6, 4, 3) == [12, 0, 0, 12, 0, 0]\n    assert simulate(bursty(30, 4, 3), 5) == (7, 0, 120)\n    assert simulate(bursty(30, 4, 3), 6) == (6, 0, 120)\n    assert simulate([], 3) == (0, 0, 0)")

### Bug hunt: A dashboard that looked fine

The dashboard shows the **mean** latency against a 300 ms target, so it reports `OK: 120.7 ms` while one request in ten takes almost a second. Report the p99 (nearest rank) instead, so it prints `SLOW: p99 is 900 ms`.

In [ ]:
import math

latencies = [12, 15, 11, 14, 13, 200, 16, 12, 14, 900] * 10
value = sum(latencies) / len(latencies)
print(f"OK: {value:.1f} ms" if value <= 300 else f"SLOW: {value:.1f} ms")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"percentiles\"\n    expect_output(\"SLOW: p99 is 900 ms\")")

### Bug hunt: A network link ten times too small

The estimate treats bytes as bits, so it says a 100 Mbit/s link has plenty of room for 10,000 requests a second of 2 KB each: `link use: 20%`. Convert bytes to bits, so it prints `link use: 160%`.

In [ ]:
qps = 10_000
bytes_per_response = 2_000
link_mbps = 100
needed_mbps = qps * bytes_per_response / 1e6
print(f"link use: {needed_mbps / link_mbps:.0%}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bytes times 8\"\n    expect_output(\"link use: 160%\")")

### Bug hunt: Planning for 100% busy

This plan divides the in-flight requests by the workers' full capacity, so it provisions exactly enough to be 100% busy at peak, where queues grow without limit: it prints `workers: 40`. Plan for 70% utilisation at peak, rounding up, so it prints `workers: 58`.

In [ ]:
import math

peak_rate = 500          # requests per second
latency_s = 0.080
in_flight = peak_rate * latency_s
print("workers:", math.ceil(in_flight))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"headroom\"\n    expect_output(\"workers: 58\")")

### Puzzle: An estimate, step by step

Arrange an estimate of peak read traffic and yearly storage for a service. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # in-flight requests at 200 requests/s taking 50 ms each
q = ___   # the p50 of [1, 2, 3, 4, 100] (nearest rank)
r = ___   # "up" or "out": adding more machines of the same kind
s = ___   # Amdahl's limit on speed-up when 90% of the work is parallel
t = ___   # True or False: a stateless service can serve any request on any instance

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10 and q == 3 and r == \"out\", \"p, q and r\"\n    assert s == 10 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a capacity plan

Write `capacity_plan(profile, measured)` for a service. `profile` has `dau`, `requests_per_user` (per day), `peak_factor`, `write_fraction` (the share of requests that write), `bytes_per_write` and `retention_days`. `measured` has `app_rps` (requests per second one app instance handles), `app_p99_ms` (its p99 latency), `db_writes_per_s` and `db_reads_per_s` (one database server's limits; reads can be spread over replicas, writes can't) and `cache_hit_rate` (the share of **reads** answered by a cache without touching the database). Plan for **70%** utilisation at peak. Return a dict: `peak_rps` (rounded to 1 place); `app_instances` (enough for the peak at 70%, and never fewer than 2, so one can fail); `app_workers_per_instance` (Little's law on that instance's share of the peak at its p99 latency, divided by 0.7, rounded up); `db_read_replicas` (database reads at peak are the reads that miss the cache; replicas needed at 70%, rounded up, at least 1); `db_writes_ok` (whether peak writes fit in one server at 70%); `storage_gb` (writes per day × bytes × retention, in 10^9 bytes, rounded to 1 place); and `warnings`: a list containing `"writes exceed one database: shard or queue them"` if `db_writes_ok` is false, and `"p99 above 300 ms"` if `app_p99_ms` is over 300.

In [ ]:
import math

def capacity_plan(profile, measured):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PROFILE = {\"dau\": 2_000_000, \"requests_per_user\": 40, \"peak_factor\": 4, \"write_fraction\": 0.05, \"bytes_per_write\": 2_000, \"retention_days\": 730}\nMEASURED = {\"app_rps\": 400, \"app_p99_ms\": 120, \"db_writes_per_s\": 2_000, \"db_reads_per_s\": 8_000, \"cache_hit_rate\": 0.9}\n\ndef test_plan():\n    \"the helpdesk at two million users\"\n    plan = capacity_plan(PROFILE, MEASURED)\n    assert plan == {\"peak_rps\": 3703.7, \"app_instances\": 14, \"app_workers_per_instance\": 46, \"db_read_replicas\": 1,\n                    \"db_writes_ok\": True, \"storage_gb\": 5840.0, \"warnings\": []}\n\ndef test_warnings():\n    \"when one database isn't enough\"\n    busy = dict(PROFILE, dau=50_000_000, write_fraction=0.2)\n    plan = capacity_plan(busy, dict(MEASURED, app_p99_ms=450, cache_hit_rate=0.5))\n    assert plan[\"peak_rps\"] == 92592.6 and plan[\"app_instances\"] == 331 and plan[\"db_read_replicas\"] == 7\n    assert plan[\"db_writes_ok\"] is False and plan[\"warnings\"] == [\"writes exceed one database: shard or queue them\", \"p99 above 300 ms\"]\n    small = capacity_plan(dict(PROFILE, dau=1_000), MEASURED)\n    assert small[\"app_instances\"] == 2 and small[\"db_read_replicas\"] == 1 and small[\"app_workers_per_instance\"] == 1")

## Warm-up

The average, 120.7 ms, describes **none** of these requests: eight took about 13 ms and two took far longer. Users don't experience averages; they experience individual requests, and the slow ones are what they remember. That's why system design measures speed with **percentiles**.

## Measuring speed

Two numbers describe how a service performs:

- **Latency**: how long one request takes, from the client's point of view.
- **Throughput**: how many requests it completes per second (often written **QPS** or **RPS**).

Latency is a **distribution**, not one number. Sort the measurements; the **p50** (median) is the value half the requests beat, **p95** the value 95% beat, **p99** the value 99% beat. The **nearest-rank** method: for the p-th percentile of `n` sorted values, take the value at position `ceil(p / 100 * n)` (counting from 1).

The **tail** (p99, p99.9) matters more than it looks:

- a page that makes **ten** backend calls waits for the **slowest**; if each call has a 1% chance of being slow, about 10% of page loads hit at least one slow call;
- the slowest requests are often from the busiest customers, with the most data;
- averages are dragged by outliers and still hide them: a p99 of 900 ms can sit under a mean of 120 ms.

Targets are written as **SLOs** (service level objectives, Week 82): "99% of requests under 300 ms, measured over 28 days".

## Back-of-the-envelope estimates

Before designing anything, estimate how big it is, with round numbers and a few minutes of arithmetic. Designs differ by orders of magnitude: 10 requests per second fits on a laptop; 100,000 needs a fleet. The usual chain:

1. **Users**: daily active users (DAU), and what each does in a day.
2. **Average rate**: actions per day divided by **86,400** seconds (about 10^5; good enough for an estimate).
3. **Peak rate**: traffic isn't flat. Multiply by a **peak factor** (2 to 10 is typical: evenings, Mondays, product launches). Design for the peak.
4. **Storage**: new data per day, times retention (and replicas). A year is about 3 x 10^7 seconds, or 365 days.
5. **Bandwidth**: requests per second times bytes per request; network speeds are quoted in **bits** per second, so multiply bytes by 8.

Useful numbers to keep in your head: reading memory takes about 100 ns; a round trip within a data centre about 0.5 ms; reading 1 MB from an SSD about 1 ms; a round trip across an ocean about 150 ms. One busy database server handles thousands to tens of thousands of simple queries a second; one app server, hundreds to a few thousand requests a second.

The read-to-write ratio (30 to 1 here) is the most important number in that calculation: read-heavy systems are scaled with caches and replicas (Days 2 and 4); write-heavy ones need different tools.

## Little's law and utilisation

**Little's law** connects three quantities for any system in a steady state: the average number of requests **in** the system `L`, the arrival rate `λ` (lambda) and the average time each spends inside `W`:

```
L = λ × W
```

At 500 requests per second with 80 ms each, there are on average `500 × 0.08 = 40` requests in progress at any moment, so the service needs at least 40 workers (threads, processes or connections) busy at once, and more for bursts. It works for anything: database connections, people in a shop, tickets in a helpdesk queue.

**Utilisation** is how busy a server is: arrival rate divided by the rate it can serve, `ρ = λ / μ`. The surprising part is what happens near 100%. For the simplest queue model (random arrivals, one server), the average time in the system is `W = 1 / (μ - λ)`:

| Utilisation | Time in system (service takes 10 ms) |
|---|---|
| 50% | 20 ms |
| 80% | 50 ms |
| 90% | 100 ms |
| 95% | 200 ms |
| 99% | 1000 ms |

The work per request hasn't changed; requests just **wait** longer, because random bursts pile up faster than a nearly-full server can clear them. That's why capacity plans aim for **50 to 70%** utilisation at peak, leaving **headroom**.

## Scaling out and load balancing

When one server isn't enough, there are two ways to grow:

- **Scale up** (vertically): a bigger machine. Simple, no code changes, but there's a largest machine, it's expensive near the top, and it's still one point of failure.
- **Scale out** (horizontally): more machines doing the same job. Nearly unlimited and resilient, but the service must be **stateless**: any instance must be able to serve any request, so session data, uploads and caches live in shared stores (a database, object storage, a shared cache), not in one instance's memory. The helpdesk is already stateless: its tokens carry everything needed, and its data is in the database.

A **load balancer** spreads requests across the instances:

- **round robin**: each in turn; simple, fine when requests cost about the same;
- **weighted round robin**: bigger machines get proportionally more;
- **least connections**: the instance with the fewest requests in progress, which adapts when some requests are slow;
- **health checks** (`/readyz`, Week 77): instances that fail them stop receiving traffic until they recover.

**Bottlenecks.** A request passes through stages (load balancer, app, cache, database), and the system can't go faster than its slowest stage, the **bottleneck**. Adding app servers does nothing if the database is the limit. **Amdahl's law** says the same for parallel work: if a fraction `p` of the work can be spread over `n` machines and the rest can't, the speed-up is `1 / ((1 - p) + p / n)`, which can never exceed `1 / (1 - p)`. With 95% parallel work, a thousand machines give less than 20 times the speed.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a service you use every day and estimate its peak requests per second from a guess at its daily users; then look for a published number to compare. Tomorrow: **caching**, the most effective (and most treacherous) way to make a read-heavy system faster.